## ERA5 → PRISM downscaling: analysis

This notebook is a compact, reproducible analysis companion to the CLI pipeline.

It covers:
1. Data visualization (ERA5 vs PRISM)
2. Model predictions vs ground truth
3. Baseline vs model comparison
4. Metrics table
5. Short observations

Assumptions:
- Data exists under `data_raw/` (see `data_pipeline/` scripts)
- Checkpoints exist under `checkpoints/` (generated by `training/train_downscaler.py`)
- Evaluation outputs are written under `results/evaluation/` (ignored by git)


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets.prism_dataset import ERA5_PRISM_Dataset
from models.baselines import upsample_latest_era5
from models.cnn_downscaler import CNNDownscaler
from models.convlstm_downscaler import ConvLSTMDownscaler

ROOT = Path('..').resolve()
ERA5_PATH = ROOT / 'data_raw' / 'era5_georgia_temp.nc'
PRISM_PATH = ROOT / 'data_raw' / 'prism'
EVAL_DIR = ROOT / 'results' / 'evaluation'

CNN_CKPT = ROOT / 'checkpoints' / 'cnn_long.pt'
CONVLSTM_CKPT = ROOT / 'checkpoints' / 'convlstm_long.pt'

DEVICE = torch.device('cpu')

def load_checkpoint_model(model_name: str, ckpt_path: Path) -> tuple[torch.nn.Module, dict]:
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    cfg = ckpt.get('model_config', {})
    if model_name == 'cnn':
        model = CNNDownscaler(
            in_channels=int(cfg['in_channels']),
            out_channels=int(cfg.get('out_channels', 1)),
            base_channels=int(cfg.get('base_channels', 32)),
        )
    elif model_name == 'convlstm':
        model = ConvLSTMDownscaler(
            input_channels=int(cfg.get('input_channels', 1)),
            hidden_channels=int(cfg.get('hidden_channels', 32)),
            out_channels=int(cfg.get('out_channels', 1)),
            kernel_size=int(cfg.get('kernel_size', 3)),
        )
    else:
        raise ValueError(model_name)

    model.load_state_dict(ckpt['model_state_dict'])
    model.to(DEVICE).eval()
    return model, ckpt


In [ ]:
# Load a small aligned dataset.
# Keep the configuration consistent with the evaluation defaults.

ds = ERA5_PRISM_Dataset(
    era5_path=str(ERA5_PATH),
    prism_path=str(PRISM_PATH),
    history_length=3,
    input_set='t2m',
    verbose=False,
)

x, y = ds[0]
print('X', tuple(x.shape), 'Y', tuple(y.shape), 'date', ds.metadata(0).date.strftime('%Y-%m-%d'))
assert torch.isfinite(x).all() and torch.isfinite(y).all()


In [ ]:
import matplotlib.pyplot as plt

# Visualize: upsampled ERA5 baseline vs PRISM target
xb = x.unsqueeze(0)
yb = y.unsqueeze(0)
era5_up = upsample_latest_era5(xb, target_size=(yb.shape[-2], yb.shape[-1])).squeeze().cpu().numpy()
prism = yb.squeeze().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)
vmin = float(min(np.min(era5_up), np.min(prism)))
vmax = float(max(np.max(era5_up), np.max(prism)))
axes[0].imshow(era5_up, cmap='coolwarm', vmin=vmin, vmax=vmax)
axes[0].set_title('ERA5 (upsampled)')
axes[0].axis('off')
axes[1].imshow(prism, cmap='coolwarm', vmin=vmin, vmax=vmax)
axes[1].set_title('PRISM target')
axes[1].axis('off')
plt.show()


In [ ]:
# Load models and run a single-sample prediction.

cnn, cnn_ckpt = load_checkpoint_model('cnn', CNN_CKPT)
convlstm, conv_ckpt = load_checkpoint_model('convlstm', CONVLSTM_CKPT)

# Use the train-split normalization stored in the checkpoint.
cnn_mean = torch.tensor(cnn_ckpt['input_norm']['mean'], device=DEVICE).view(1, 1, -1, 1, 1)
cnn_std = torch.tensor(cnn_ckpt['input_norm']['std'], device=DEVICE).view(1, 1, -1, 1, 1).clamp(min=1e-6)

cl_mean = torch.tensor(conv_ckpt['input_norm']['mean'], device=DEVICE).view(1, 1, -1, 1, 1)
cl_std = torch.tensor(conv_ckpt['input_norm']['std'], device=DEVICE).view(1, 1, -1, 1, 1).clamp(min=1e-6)

xb = x.unsqueeze(0).to(DEVICE)
yb = y.unsqueeze(0).to(DEVICE)

with torch.no_grad():
    cnn_pred = cnn((xb - cnn_mean) / cnn_std, target_size=(yb.shape[-2], yb.shape[-1]))
    cl_pred = convlstm((xb - cl_mean) / cl_std, target_size=(yb.shape[-2], yb.shape[-1]))

cnn_pred_np = cnn_pred.squeeze().cpu().numpy()
cl_pred_np = cl_pred.squeeze().cpu().numpy()
target_np = yb.squeeze().cpu().numpy()

assert np.isfinite(cnn_pred_np).all() and np.isfinite(cl_pred_np).all()


In [ ]:
# Compare baselines vs models on a single example.

abs_err_cnn = np.abs(cnn_pred_np - target_np)
abs_err_cl = np.abs(cl_pred_np - target_np)

fig, axes = plt.subplots(2, 3, figsize=(12, 7.5), constrained_layout=True)

# Row 1: fields
vmin = float(min(np.min(era5_up), np.min(cnn_pred_np), np.min(cl_pred_np), np.min(target_np)))
vmax = float(max(np.max(era5_up), np.max(cnn_pred_np), np.max(cl_pred_np), np.max(target_np)))

axes[0, 0].imshow(era5_up, cmap='coolwarm', vmin=vmin, vmax=vmax)
axes[0, 0].set_title('Baseline: ERA5 upsampled')
axes[0, 0].axis('off')

axes[0, 1].imshow(cnn_pred_np, cmap='coolwarm', vmin=vmin, vmax=vmax)
axes[0, 1].set_title('CNN prediction')
axes[0, 1].axis('off')

axes[0, 2].imshow(cl_pred_np, cmap='coolwarm', vmin=vmin, vmax=vmax)
axes[0, 2].set_title('ConvLSTM prediction')
axes[0, 2].axis('off')

# Row 2: errors + target
err_max = float(max(np.max(abs_err_cnn), np.max(abs_err_cl)))
axes[1, 0].imshow(target_np, cmap='coolwarm', vmin=vmin, vmax=vmax)
axes[1, 0].set_title('PRISM target')
axes[1, 0].axis('off')

axes[1, 1].imshow(abs_err_cnn, cmap='magma', vmin=0.0, vmax=err_max)
axes[1, 1].set_title('|CNN error|')
axes[1, 1].axis('off')

axes[1, 2].imshow(abs_err_cl, cmap='magma', vmin=0.0, vmax=err_max)
axes[1, 2].set_title('|ConvLSTM error|')
axes[1, 2].axis('off')

plt.show()


In [ ]:
# Metrics table from the latest evaluation output.

csv_path = EVAL_DIR / 'baselines_summary.csv'
if not csv_path.exists():
    raise FileNotFoundError(f"Missing {csv_path}. Run evaluation/evaluate_model.py first.")

df = pd.read_csv(csv_path)
df = df[['model', 'rmse', 'mae', 'bias', 'correlation']].sort_values('rmse')
df


## Short observations

- With `t2m`-only inputs, the **persistence (upsampled ERA5)** baseline is strong.
- In the example run recorded in `results/evaluation/baselines_summary.csv`, **ConvLSTM improves over persistence** on RMSE/MAE.
- The CNN model may not consistently beat persistence on this dataset/configuration; this is worth revisiting once additional ERA5 predictors are available.


## Interpretation notes

- CNN often struggles with temporal dependencies because the ERA5 history is treated as stacked channels rather than a sequence.
- ConvLSTM is better suited to sequence inputs because it updates a hidden state across timesteps and can represent short-term temporal evolution.
- In error maps, larger errors typically appear in regions with strong spatial gradients (e.g., sharp transitions), where small spatial/phase differences are amplified.


In [ ]:
# Experiment summary table (generated by scripts/run_core_experiments.py)

exp_summary = ROOT / 'results' / 'experiments' / 'summary.csv'
if exp_summary.exists():
    exp_df = pd.read_csv(exp_summary)
    exp_df
else:
    print(f"Missing {exp_summary}. Run scripts/run_core_experiments.py first.")


In [ ]:
# RMSE vs history length (when summary.csv is available)

if 'exp_df' in globals() and isinstance(exp_df, pd.DataFrame) and not exp_df.empty:
    fig, ax = plt.subplots(1, 1, figsize=(7.5, 4.2), constrained_layout=True)
    for (model, input_set), grp in exp_df.groupby(['model', 'input_set']):
        if model not in ('cnn', 'convlstm', 'persistence'):
            continue
        g = grp.sort_values('history')
        ax.plot(g['history'], g['rmse'], marker='o', label=f"{model}-{input_set}")
    ax.set_xlabel('history length')
    ax.set_ylabel('RMSE')
    ax.set_title('RMSE vs history length')
    ax.grid(alpha=0.25)
    ax.legend(ncol=2, fontsize=9)
    plt.show()


## Experiment observations (to update after running core4)

- Start with `t2m` runs to validate the pipeline end-to-end.
- When `core4` data is available, compare `t2m` vs `core4` at the same history length and split.
- Focus on delta vs persistence and whether gains saturate with longer history.
